# Exercise lab 4

We will use two models: one instruct-tuned and one not.

In [2]:

import torch


GENERATIVE_MODEL = "meta-llama/Llama-3.2-1B" # Login required
INSTRUCT_MODEL = "meta-llama/Llama-3.2-1B-Instruct" # Login required
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

To use llama models:
- Accept the license in huggingface: https://huggingface.co/meta-llama/Llama-3.2-1B and request access
- Create a token in your huggingface account https://huggingface.co/settings/tokens
- Use it with the command:
    ```bash
    uvx hf auth login
    ```

In [35]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from transformers.generation.utils import GenerateDecoderOnlyOutput, GenerateBeamDecoderOnlyOutput
import torch
from torch import nn
import torch.nn.functional as F

def describe_model(model: AutoModelForCausalLM):
    n_params = model.num_parameters()
    print(f"Total parameters: {n_params:,}")
    print(f"Architecture: {model.config.architectures[0]}")
    try:
        ctx = model.config.max_position_embeddings
    except:
        ctx = model.config.seq_length
    print(f"Maximum context length: {ctx:,} tokens")
    print(f"Parameters dtype: {model.config.torch_dtype}")
    print(f"Vocabulary size: {model.config.vocab_size:,} tokens")

streamer = TextStreamer(None, skip_prompt=True, skip_special_tokens=True)
gen_kwargs = {
    "max_new_tokens": 200,
    "do_sample": True
}

def create_prompt(tokenizer, message):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": message},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return prompt


def generate_response(model, tokenizer, prompt, streamer=streamer, gen_kwargs=gen_kwargs, return_scores=True, device=DEVICE):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device) # return_tensors="pt" for PyTorch
    with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                streamer=streamer,
                return_dict_in_generate=True,
                output_scores=return_scores,
                **gen_kwargs
            )
    return output_ids

def inspect_token_probabilities(response, tokenizer, top_k=5, min_prob=0.01):
    """Show top token probabilities per generated token.

    Args:
        response: model.generate output with output_scores=True
        tokenizer: Hugging Face tokenizer
        top_k: int, max number of top tokens to show
        min_prob: float, minimum probability threshold to display
    """
    scores = response.scores
    sequences = response.sequences[0]
    generated_ids = sequences[-len(scores):]
    special_tokens = tokenizer.all_special_ids

    for token_id, logits in zip(generated_ids, scores):
        if token_id.item() in special_tokens:
            continue  # Skip special tokens
        logits = logits[0] if logits.dim() == 2 else logits
        probs = F.softmax(logits, dim=-1)

        token_str = tokenizer.decode([token_id])
        token_prob = probs[token_id].item() * 100

        top_probs, top_ids = torch.topk(probs, top_k)
        filtered = [(p, idx) for p, idx in zip(top_probs, top_ids) if p.item() >= min_prob]

        print(f"Token: {token_str!r} (generated) | Prob: {token_prob:.2f}%")
        for p, idx in filtered:
            cand = tokenizer.decode([idx.item()])
            print(f"   {cand!r}: {p.item() * 100:.2f}%")
        print()

In [10]:
if DEVICE == "cuda:0":
    mem0 = torch.cuda.memory_allocated(0) / 1e9
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)
model = AutoModelForCausalLM.from_pretrained(INSTRUCT_MODEL).to(DEVICE)
if DEVICE == "cuda:0":
    mem1 = torch.cuda.memory_allocated(0) / 1e9
    print(f"GPU memory used by model: {mem1 - mem0:.2f} GB")
describe_model(model)

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 1504.97it/s, Materializing param=model.norm.weight]                              


Total parameters: 1,235,814,400
Architecture: LlamaForCausalLM
Maximum context length: 131,072 tokens
Parameters dtype: torch.bfloat16
Vocabulary size: 128,256 tokens


In [45]:
torch.manual_seed(42)  # For reproducibility

tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)
model = AutoModelForCausalLM.from_pretrained(INSTRUCT_MODEL).to(DEVICE)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
gen_kwargs = {
    "max_new_tokens": 100,
}
msg = "Give me ideas for a birthday party for a 10 year old child."
response = generate_response(model, tokenizer, msg, streamer=streamer, gen_kwargs=gen_kwargs, device=DEVICE)

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 1320.97it/s, Materializing param=model.norm.weight]                              
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 Fun and exciting, but not too scary or intense.
Here are a few ideas for a 10 year old's birthday party:
1. **Backyard Camping Adventure**: Set up a tent in the backyard and have a camping adventure with s'mores, a campfire, and a nighttime scavenger hunt.
2. **Superhero Training Academy**: Create a superhero training course with obstacles, challenges, and games that test the kids' superhero skills.
3. **Minecraft Party**: Host a Minecraft


It's a generative model that has not been fine-tuned for following instructions, so it may not respond as expected to direct commands or questions.

In [37]:
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)
model = AutoModelForCausalLM.from_pretrained(INSTRUCT_MODEL).to(DEVICE)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
prompt = create_prompt(tokenizer, "Explain the theory of relativity in simple terms.")
print(prompt)

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 1280.21it/s, Materializing param=model.norm.weight]                              


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 02 Feb 2026

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Explain the theory of relativity in simple terms.<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [38]:
response = generate_response(model, tokenizer, prompt, streamer=streamer, gen_kwargs=gen_kwargs, device=DEVICE)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


The theory of relativity! One of the most mind-bending concepts in physics. I'd be happy to break it down in simple terms.

**What is the theory of relativity?**

The theory of relativity, developed by Albert Einstein, is a fundamental concept in modern physics that explains how the universe works. It's a bit like a puzzle, and I'll try to explain it in a way that's easy to understand.

**The Basics: Time and Space**

Imagine you're on a train, and you throw a ball straight up in the air. What happens? The ball comes down and lands in your hand, right? Now, imagine your friend is standing outside the train, watching you throw the ball. From their perspective, the ball doesn't just go straight up and down - it also moves forward, because the train is moving really fast.

**Special Relativity**

Einstein said that how we measure time and space is relative, not absolute. This means that how we


Usage of no convertational model could be for completing text, generating code, or other tasks where following specific instructions is not required.

In [43]:
tokenizer = AutoTokenizer.from_pretrained(GENERATIVE_MODEL)
model = AutoModelForCausalLM.from_pretrained(GENERATIVE_MODEL).to(DEVICE)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
gen_kwargs = {
    "max_new_tokens": 50,
}
msg = f"""
    --- POEMA ---
    Al olmo viejo, hendido por el rayo
    y en su mitad podrido,
    con las lluvias de abril y el sol de mayo
    algunas hojas verdes le han salido.
"""
response = generate_response(model, tokenizer, msg, streamer=streamer, gen_kwargs=gen_kwargs, device=DEVICE)

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 1217.60it/s, Materializing param=model.norm.weight]                              
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


    Al olmo viejo, hendido por el rayo
    y en su mitad podrido,
    con las lluvias de abril y el sol de mayo
    algunas hojas verdes le han salido.
    Al


While the instruct-tuned model will talk about the poem, the generative model will try to continue it.

In [44]:
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)
model = AutoModelForCausalLM.from_pretrained(INSTRUCT_MODEL).to(DEVICE)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
gen_kwargs = {
    "max_new_tokens": 50,
}
msg = f"""
    --- POEMA ---
    Al olmo viejo, hendido por el rayo
    y en su mitad podrido,
    con las lluvias de abril y el sol de mayo
    algunas hojas verdes le han salido.
"""
prompt = create_prompt(tokenizer, msg)
response = generate_response(model, tokenizer, prompt, streamer=streamer, gen_kwargs=gen_kwargs, device=DEVICE)

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 1348.56it/s, Materializing param=model.norm.weight]                              
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


¡Un poema hermoso!

La imagen que presentas es una metáfora muy poderosa. La comparación de "olmo viejo" con "podrido" es especialmente interesante, ya que sugiere una transformación
